# Fraud Detection Notebook
## Focus: Case Management & Homeowner Assistance Fund (HAF)

This notebook contains simple, practical fraud detection patterns for:
- Case management (document reuse, staff behavior)
- Homeowner Assistance Fund (HAF) applications (duplicate properties, identity reuse, income mismatch)


In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 1. Case Management Fraud
Fraud patterns:
- Reuse of the same document across multiple cases
- Same contact details across different identities
- Staff behavior anomalies (one reviewer approving many risky cases)

Below are simple examples to detect these patterns.

### 1.1 Document Reuse Across Cases
We assume each uploaded document is hashed (e.g., SHA-256) and stored as `doc_hash`.
If the same `doc_hash` appears in multiple cases, it may indicate document reuse or fraud rings.

In [2]:
docs = pd.DataFrame({
    "case_id":   [101, 102, 103, 104, 105, 106],
    "doc_hash":  ["abc123", "xyz999", "abc123", "lmn555", "xyz999", "pqr777"],
    "doc_type":  ["income", "id", "income", "mortgage", "id", "hardship"]
})

docs


,case_id,doc_hash,doc_type
0,101,abc123,income
1,102,xyz999,id
2,103,abc123,income
3,104,lmn555,mortgage
4,105,xyz999,id
5,106,pqr777,hardship


In [3]:
# Find documents reused across multiple cases
duplicate_docs = docs[docs.duplicated("doc_hash", keep=False)].sort_values("doc_hash")
duplicate_docs

# Group by doc_hash to see how many cases share the same document
duplicate_docs.groupby("doc_hash")["case_id"].apply(list).reset_index(name="case_ids")

,doc_hash,case_ids
0,abc123,"[101, 103]"
1,xyz999,"[102, 105]"


### 1.2 Shared Contact Details Across Different Identities
If multiple cases with different names share the same phone or email, it may indicate synthetic identities or coordinated fraud.

In [4]:
cases = pd.DataFrame({
    "case_id":   [201, 202, 203, 204, 205],
    "name":      ["John Smith", "Jon Smyth", "Mary Lee", "Alex Doe", "J. Smith"],
    "phone":     ["555-1111", "555-1111", "555-2222", "555-3333", "555-1111"],
    "email":     ["john@example.com", "john.s@example.com", "mary@example.com", "alex@example.com", "john@example.com"]
})

cases


,case_id,name,phone,email
0,201,John Smith,555-1111,john@example.com
1,202,Jon Smyth,555-1111,john.s@example.com
2,203,Mary Lee,555-2222,mary@example.com
3,204,Alex Doe,555-3333,alex@example.com
4,205,J. Smith,555-1111,john@example.com


In [ ]:
# Cases sharing the same phone number
phone_dupes = cases[cases.duplicated("phone", keep=False)].sort_values("phone")
phone_dupes

# Cases sharing the same email
email_dupes = cases[cases.duplicated("email", keep=False)].sort_values("email")
email_dupes

# Group by phone to see clusters
phone_clusters = phone_dupes.groupby("phone")["case_id"].apply(list).reset_index(name="case_ids")
phone_clusters

### 1.3 Reviewer / Staff Anomalies
If one reviewer approves a disproportionate number of cases, especially high-risk ones, it may indicate internal collusion or control weaknesses.


In [5]:
reviews = pd.DataFrame({
    "case_id":     [301,302,303,304,305,306,307,308,309,310],
    "reviewer_id": [1,1,1,2,2,3,3,3,3,3],
    "risk_score":  [80,75,90,40,35,85,88,82,20,25],  # 80+ considered high risk
    "decision":    ["approved","approved","approved","approved","declined","approved","approved","approved","declined","declined"]
})

reviews


,case_id,reviewer_id,risk_score,decision
0,301,1,80,approved
1,302,1,75,approved
2,303,1,90,approved
3,304,2,40,approved
4,305,2,35,declined
5,306,3,85,approved
6,307,3,88,approved
7,308,3,82,approved
8,309,3,20,declined
9,310,3,25,declined


In [6]:
# Count approvals per reviewer
approvals = reviews[reviews["decision"] == "approved"].groupby("reviewer_id")["case_id"].count().reset_index(name="approved_count")
approvals

# High-risk approvals per reviewer
high_risk_approvals = reviews[(reviews["decision"] == "approved") & (reviews["risk_score"] >= 80)] \
    .groupby("reviewer_id")["case_id"].count().reset_index(name="high_risk_approved")
high_risk_approvals

# Merge to see overall picture
reviewer_stats = approvals.merge(high_risk_approvals, on="reviewer_id", how="left").fillna(0)
reviewer_stats["high_risk_ratio"] = reviewer_stats["high_risk_approved"] / reviewer_stats["approved_count"]
reviewer_stats.sort_values("high_risk_ratio", ascending=False)

,reviewer_id,approved_count,high_risk_approved,high_risk_ratio
2,3,3,3.0,1.000000
0,1,3,2.0,0.666667
1,2,1,0.0,0.000000


# 2. Homeowner Assistance Fund (HAF) Fraud
Fraud patterns:
- Multiple applications for the same property
- Same identity (SSN) across multiple applications
- Income misrepresentation vs. bank deposits

Below are examples to detect these patterns.

### 2.1 Multiple Applications for the Same Property
If the same property address appears in multiple applications, it may indicate duplicate claims or coordinated fraud.


In [7]:
haf_apps = pd.DataFrame({
    "application_id":   [1,2,3,4,5,6],
    "property_address": ["123 Main St","123 Main St","55 Oak Rd","77 Pine St","55 Oak Rd","999 Elm St"],
    "ssn":              ["111-11-1111","222-22-2222","111-11-1111","333-33-3333","444-44-4444","555-55-5555"],
    "state":            ["TX","TX","TX","CA","TX","CA"]
})

haf_apps


,application_id,property_address,ssn,state
0,1,123 Main St,111-11-1111,TX
1,2,123 Main St,222-22-2222,TX
2,3,55 Oak Rd,111-11-1111,TX
3,4,77 Pine St,333-33-3333,CA
4,5,55 Oak Rd,444-44-4444,TX
5,6,999 Elm St,555-55-5555,CA


In [8]:
# Duplicate property addresses
dup_properties = haf_apps[haf_apps.duplicated("property_address", keep=False)].sort_values("property_address")
dup_properties

# Group by property to see all applications per address
property_clusters = dup_properties.groupby("property_address")["application_id"].apply(list).reset_index(name="application_ids")
property_clusters

,property_address,application_ids
0,123 Main St,"[1, 2]"
1,55 Oak Rd,"[3, 5]"


### 2.2 Identity Reuse Across Applications
If the same SSN appears in multiple applications (especially across states), it may indicate abuse of the program or identity fraud.


In [9]:
# Applications with the same SSN
dup_ssn = haf_apps[haf_apps.duplicated("ssn", keep=False)].sort_values("ssn")
dup_ssn

# Group by SSN to see all applications per identity
ssn_clusters = dup_ssn.groupby("ssn")[["application_id","state"]].apply(lambda x: x.to_dict("records")).reset_index(name="applications")
ssn_clusters

,ssn,applications
0,111-11-1111,"[{'application_id': 1, 'state': 'TX'}, {'appli..."


### 2.3 Income vs. Bank Deposits Mismatch
If claimed income is significantly lower than actual deposits, it may indicate misrepresentation to qualify for assistance.


In [10]:
income_data = pd.DataFrame({
    "application_id":        [1,2,3,4,5],
    "claimed_monthly_income": [3000, 4000, 2500, 5000, 3200],
    "avg_monthly_deposits":   [8000, 3500, 2600, 5200, 9000]
})

income_data


,application_id,claimed_monthly_income,avg_monthly_deposits
0,1,3000,8000
1,2,4000,3500
2,3,2500,2600
3,4,5000,5200
4,5,3200,9000


In [11]:
# Flag applications where deposits are much higher than claimed income
threshold_multiplier = 1.5
income_data["income_mismatch"] = income_data["avg_monthly_deposits"] > (income_data["claimed_monthly_income"] * threshold_multiplier)
income_data

# Show only suspicious cases
suspicious_income = income_data[income_data["income_mismatch"]]
suspicious_income

,application_id,claimed_monthly_income,avg_monthly_deposits,income_mismatch
0,1,3000,8000,True
4,5,3200,9000,True


# 3. Combining Signals into a Simple Fraud Risk View
In a real system, you would combine multiple signals (document reuse, identity reuse, income mismatch, etc.) into a risk score or rule-based engine.
Below is a simple example combining a few flags for HAF applications.

In [18]:
# Merge property and SSN duplicate flags into the HAF dataset
haf = haf_apps.copy()

haf["dup_property"] = haf["property_address"].isin(dup_properties["property_address"])
haf["dup_ssn"] = haf["ssn"].isin(dup_ssn["ssn"])

# Add income mismatch info
haf = haf.merge(
    income_data[["application_id", "income_mismatch"]],
    on="application_id",
    how="left"
)

# Ensure boolean dtype explicitly (avoids FutureWarning)
haf["income_mismatch"] = (
    haf["income_mismatch"]
    .astype("boolean")
    .fillna(False)
)

# Compute fraud risk score
haf["fraud_risk_score"] = (
    haf["dup_property"].astype(int) * 40 +
    haf["dup_ssn"].astype(int) * 40 +
    haf["income_mismatch"].astype(int) * 30
)

haf.sort_values("fraud_risk_score", ascending=False)

,application_id,property_address,ssn,state,dup_property,dup_ssn,income_mismatch,fraud_risk_score
0,1,123 Main St,111-11-1111,TX,True,True,True,110
2,3,55 Oak Rd,111-11-1111,TX,True,True,False,80
4,5,55 Oak Rd,444-44-4444,TX,True,False,True,70
1,2,123 Main St,222-22-2222,TX,True,False,False,40
3,4,77 Pine St,333-33-3333,CA,False,False,False,0
5,6,999 Elm St,555-55-5555,CA,False,False,False,0
